In [ ]:
import os
import re
import pandas as pd
from datasets import Dataset
from transformers import T5ForConditionalGeneration, T5Tokenizer, Trainer, TrainingArguments
import torch

def neutralize_text(sentence):
    sentence = re.sub(r"\[(he|she)\]", "[they]", sentence, flags=re.IGNORECASE)
    sentence = re.sub(r"\[(his|her)\]", "[their]", sentence, flags=re.IGNORECASE)
    sentence = re.sub(r"\[(him|her)\]", "[them]", sentence, flags=re.IGNORECASE)

    sentence = re.sub(r"(\[they\])\s+is\b", r"\1 are", sentence, flags=re.IGNORECASE)
    sentence = re.sub(r"(\[they\])\s+has\b", r"\1 have", sentence, flags=re.IGNORECASE)
    sentence = re.sub(r"(\[they\])\s+was\b", r"\1 were", sentence, flags=re.IGNORECASE)

    return sentence

tsv_file = "../data/WinoBias/new/TSV/anti_stereotyped_type1.dev.tsv"
df = pd.read_csv(tsv_file, sep="\t")


df['label'] = pd.to_numeric(df['label'], errors='coerce')

stereotypical_df = df[df['label'] == 1].copy()

stereotypical_df["target"] = stereotypical_df["sentence"].apply(neutralize_text)

paired_examples = [
    {"source": row["sentence"], "target": row["target"]}
    for _, row in stereotypical_df.iterrows()
]

dataset = Dataset.from_list(paired_examples)


model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)


def preprocess_function(examples):
    inputs = ["paraphrase: " + ex for ex in examples["source"]]
    targets = examples["target"]
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length")
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=128, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = dataset.map(preprocess_function, batched=True)


training_args = TrainingArguments(
    output_dir="../models/gender_neutral_finetuned_model",
    per_device_train_batch_size=4,
    num_train_epochs=3,
    logging_steps=10,
    save_steps=100,
    learning_rate=5e-5,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

trainer.train()

model.save_pretrained("../models/gender_neutral_finetuned_model")
tokenizer.save_pretrained("../models/gender_neutral_finetuned_model")


Index(['1',
       '[The developer] argued with the designer because [she] did not like the design.',
       '0'],
      dtype='object')


KeyError: 'label'

In [30]:
import os
import re
import pandas as pd
from datasets import Dataset
from transformers import T5ForConditionalGeneration, T5Tokenizer, Trainer, TrainingArguments

# === 1. Rule-based Gender-Neutralizer ===
def neutralize_text(sentence):
    # Pronoun substitutions
    sentence = re.sub(r"\[(he|she)\]", "[they]", sentence, flags=re.IGNORECASE)
    sentence = re.sub(r"\[(his|her)\]", "[their]", sentence, flags=re.IGNORECASE)
    sentence = re.sub(r"\[(him|her)\]", "[them]", sentence, flags=re.IGNORECASE)

    # Verb agreement adjustments
    sentence = re.sub(r"(\[they\])\s+is\b", r"\1 are", sentence, flags=re.IGNORECASE)
    sentence = re.sub(r"(\[they\])\s+has\b", r"\1 have", sentence, flags=re.IGNORECASE)
    sentence = re.sub(r"(\[they\])\s+was\b", r"\1 were", sentence, flags=re.IGNORECASE)

    return sentence

# === 2. Load & Process Dataset ===
tsv_file = "data.tsv"  # <-- your TSV file
df = pd.read_csv(tsv_file, sep="\t")

# Ensure numeric labels
df['label'] = pd.to_numeric(df['label'], errors='coerce')

# Filter only stereotypical sentences (label == 1)
stereotypical_df = df[df['label'] == 1].copy()

# Apply neutralization to generate targets
stereotypical_df["target"] = stereotypical_df["sentence"].apply(neutralize_text)

# Create source-target training pairs
paired_examples = [
    {"source": row["sentence"], "target": row["target"]}
    for _, row in stereotypical_df.iterrows()
]

# === 3. Convert to Hugging Face Dataset ===
dataset = Dataset.from_list(paired_examples)

# === 4. Load Model & Tokenizer ===
model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

# === 5. Preprocess Text for Model ===
def preprocess_function(examples):
    inputs = ["paraphrase: " + ex for ex in examples["source"]]
    targets = examples["target"]
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length")
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=128, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = dataset.map(preprocess_function, batched=True)

# === 6. Training Configuration ===
training_args = TrainingArguments(
    output_dir="./gender_neutral_finetuned_model",
    per_device_train_batch_size=4,
    num_train_epochs=3,
    logging_steps=10,
    save_steps=100,
    learning_rate=5e-5,
    fp16=True,  # Mixed precision for RTX 4060 (8GB VRAM)
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

# === 7. Fine-tune the Model ===
trainer.train()

# === 8. Save Model & Tokenizer ===
model.save_pretrained("./gender_neutral_finetuned_model")
tokenizer.save_pretrained("./gender_neutral_finetuned_model")


Original: [The developer] argued with the designer because [she] did not like the design and [he] is busy.
Neutralized: [The developer] argued with the designer because [they] did not like the design and [they] are busy.
